# NB37 — Multi-Level Environmental Niche Breadth Analysis

**Hypothesis:** Metal-gene density (ko_per_mb_primary) is specifically driven by
*geochemical* niche breadth more than by general habitat or climate niche breadth.

| Level | Description | Source |
|-------|-------------|--------|
| 0a | Habitat niche (MA Levins B_std, P1 reference) | MicrobeAtlas |
| 0b | Habitat niche (EMP Levins B_std, NB08 approach) | EMP |
| 1  | Edaphic niche (pH SD + temp SD) | MA × GeoROC spatial join |
| 2  | Global geochemical niche (9 GeoROC metal SDs) | MA × GeoROC spatial join |
| 3  | Combined global niche (edaphic + geochemical) | Levels 1+2 |
| 4  | USA geochemical niche (8 USGS NGSA metal SDs) | MA × NGSA |
| 5  | Climate niche (WorldClim BIO1/BIO12) | BERDL optional |
| 6  | Land cover niche (ESA CCI categories) | BERDL optional |

**Expected:** β(geochemical niche) < 0 ≈ P1 ref (β = −0.0207). Climate niche should be non-significant.


In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
warnings.filterwarnings("ignore")

PROJECT = Path("/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology")
DATA    = PROJECT / "data"
FIGS    = PROJECT / "figures"

sys.path.insert(0, "/home/hmacgregor/BERIL-research-observatory/tools")
sys.path.insert(0, str(PROJECT / "scripts"))
from figure_style import apply_style, save, PALETTE, FIGW, ROW_H, grid_h
apply_style()
from pgls_utils import run_pgls

TREE_BAC = DATA / "gtdb_bac_genus_pruned.tree"
assert TREE_BAC.exists(), f"Tree not found: {TREE_BAC}"

P1_BETA = -0.0207; P1_SE = 0.00368; P1_N = 1574

_SPARK_AVAILABLE = False; _spark = None
try:
    from berdl_utils import get_spark_session
    _spark = get_spark_session()
    _SPARK_AVAILABLE = True
    print("Spark: connected")
except BaseException as _e:
    print(f"Spark unavailable (levels 5-6 will be skipped): {_e}")


In [ ]:
p1 = pd.read_csv(DATA / "01_pgls_input_bacteria.csv")
p1["genus_lower"] = p1["genus_lower"].str.lower().str.strip()
print(f"P1 genera: {len(p1)}")

def merge_p1(df, on="genus_lower"):
    m = p1.merge(df, on=on, how="inner")
    return m.dropna(subset=["ko_per_mb_primary", "mean_levins_B_std"])

def z_score(s):
    v = s.dropna(); return (s - v.mean()) / v.std()

def pca1(df, cols):
    sub = df[cols].copy()
    mask = sub.notna().any(axis=1)
    sub  = sub[mask].fillna(sub.median())
    sc   = StandardScaler().fit_transform(sub)
    pca  = PCA(n_components=1)
    pc1  = pca.fit_transform(sc)[:, 0]
    out  = pd.Series(np.nan, index=df.index)
    out.loc[sub.index] = pc1
    return out, pca.explained_variance_ratio_[0]

def extract_pgls(res, label):
    if res is None:
        return dict(label=label, beta=np.nan, SE=np.nan, p=np.nan, n=np.nan, lam=np.nan)
    betas = res.get("betas", {}) or {}
    SEs   = res.get("SEs", {}) or {}
    ps    = res.get("p_values", {}) or {}
    key   = list(betas.keys())[0] if betas else "niche"
    return dict(
        label = label,
        beta  = betas.get(key, res.get("beta", np.nan)),
        SE    = SEs.get(key, res.get("SE", np.nan)),
        p     = ps.get(key, res.get("p_value", np.nan)),
        n     = res.get("n", np.nan),
        lam   = res.get("lambda_est", np.nan),
    )


## Level 0: Habitat niche (reference)

In [ ]:
# 0a: MicrobeAtlas Levins B (= P1 primary result, confirming it runs)
df0a = p1.dropna(subset=["ko_per_mb_primary","mean_levins_B_std"]).copy()
df0a["niche"] = z_score(df0a["mean_levins_B_std"])
df0a = df0a.dropna(subset=["niche"])
res0a = run_pgls(df0a, str(TREE_BAC), response="ko_per_mb_primary",
                 predictors=["niche"], taxon_col="genus_lower",
                 label="L0a_MA_levins", min_n=100)
r0a = extract_pgls(res0a, "L0a: MA Levins B (P1 reference)")
print(f"L0a: beta={r0a['beta']:+.5f}  p={r0a['p']:.3e}  n={r0a['n']}")

# 0b: EMP Levins B
emp = pd.read_csv(DATA / "emp_niche_pgls_input.csv")
emp["genus_lower"] = emp["genus_lower"].str.lower().str.strip()
df0b = merge_p1(emp[["genus_lower","emp_levins_B_std"]].dropna())
df0b["niche"] = z_score(df0b["emp_levins_B_std"])
df0b = df0b.dropna(subset=["niche"])
res0b = run_pgls(df0b, str(TREE_BAC), response="ko_per_mb_primary",
                 predictors=["niche"], taxon_col="genus_lower",
                 label="L0b_EMP_levins", min_n=30)
r0b = extract_pgls(res0b, "L0b: EMP Levins B (NB08)")
print(f"L0b: beta={r0b['beta']:+.5f}  p={r0b['p']:.3e}  n={r0b['n']}")


## Levels 1-3: Global environmental niche (GeoROC + edaphic)

In [ ]:
env_global = pd.read_csv(DATA / "env_niche_global_spark.csv")
env_global["genus_lower"] = env_global["genus_lower"].str.lower().str.strip()
EDAPHIC = [c for c in ["pH_sd","temp_sd"] if c in env_global.columns]
GEOCHEM  = [c for c in env_global.columns if c.startswith("georoc_") and c.endswith("_sd")]
print(f"env_global: {len(env_global)} genera; edaphic={EDAPHIC}; geochem={GEOCHEM}")

# Level 1: edaphic
df1 = merge_p1(env_global[["genus_lower"] + EDAPHIC].dropna(subset=EDAPHIC, how="any"))
df1["niche_pc1"], var1 = pca1(df1, EDAPHIC)
df1["niche"] = z_score(df1["niche_pc1"]); df1 = df1.dropna(subset=["niche"])
res1 = run_pgls(df1, str(TREE_BAC), response="ko_per_mb_primary",
                predictors=["niche"], taxon_col="genus_lower",
                label="L1_edaphic", min_n=30)
r1 = extract_pgls(res1, f"L1: Edaphic niche (pH+temp, {var1*100:.0f}% PC1 var)")
print(f"L1: beta={r1['beta']:+.5f}  p={r1['p']:.3e}  n={r1['n']}")

# Level 2: GeoROC geochemical
df2 = merge_p1(env_global[["genus_lower"] + GEOCHEM].dropna(subset=GEOCHEM, how="all"))
df2["niche_pc1"], var2 = pca1(df2, GEOCHEM)
df2["niche"] = z_score(df2["niche_pc1"]); df2 = df2.dropna(subset=["niche"])
res2 = run_pgls(df2, str(TREE_BAC), response="ko_per_mb_primary",
                predictors=["niche"], taxon_col="genus_lower",
                label="L2_geochem_global", min_n=30)
r2 = extract_pgls(res2, f"L2: Global geochemical niche (GeoROC, {var2*100:.0f}% PC1 var)")
print(f"L2: beta={r2['beta']:+.5f}  p={r2['p']:.3e}  n={r2['n']}")

# GeoROC PC loadings
sub_g = env_global.set_index("genus_lower")[GEOCHEM].dropna(how="all").fillna(env_global[GEOCHEM].median())
sc_g  = StandardScaler().fit_transform(sub_g)
pca_g = PCA(n_components=2).fit(sc_g)
print(f"GeoROC PCA var: {pca_g.explained_variance_ratio_}")
print("PC1 loadings:", dict(zip([c.replace("georoc_","").replace("_sd","") for c in GEOCHEM], pca_g.components_[0].round(3))))

# Level 3: combined
ALL_COLS = EDAPHIC + GEOCHEM
df3 = merge_p1(env_global[["genus_lower"] + ALL_COLS].dropna(subset=ALL_COLS, how="all"))
df3["niche_pc1"], var3 = pca1(df3, ALL_COLS)
df3["niche"] = z_score(df3["niche_pc1"]); df3 = df3.dropna(subset=["niche"])
res3 = run_pgls(df3, str(TREE_BAC), response="ko_per_mb_primary",
                predictors=["niche"], taxon_col="genus_lower",
                label="L3_combined", min_n=30)
r3 = extract_pgls(res3, f"L3: Combined global niche ({var3*100:.0f}% PC1 var)")
print(f"L3: beta={r3['beta']:+.5f}  p={r3['p']:.3e}  n={r3['n']}")


## Level 4: USA geochemical niche (USGS NGSA)

In [ ]:
env_ngsa = pd.read_csv(DATA / "env_niche_ngsa_spark.csv")
env_ngsa["genus_lower"] = env_ngsa["genus_lower"].str.lower().str.strip()
NGSA_ICP = [c for c in env_ngsa.columns if c.endswith("_sd") and "ICP" in c and "MMI" not in c]
print(f"NGSA ICP-MS columns: {NGSA_ICP}")

df4 = merge_p1(env_ngsa[["genus_lower"] + NGSA_ICP].dropna(subset=NGSA_ICP, how="all"))
df4["niche_pc1"], var4 = pca1(df4, NGSA_ICP)
df4["niche"] = z_score(df4["niche_pc1"]); df4 = df4.dropna(subset=["niche"])
res4 = run_pgls(df4, str(TREE_BAC), response="ko_per_mb_primary",
                predictors=["niche"], taxon_col="genus_lower",
                label="L4_usa_geochem", min_n=30)
r4 = extract_pgls(res4, f"L4: USA geochemical niche (NGSA, {var4*100:.0f}% PC1 var)")
print(f"L4: beta={r4['beta']:+.5f}  p={r4['p']:.3e}  n={r4['n']}")


## Levels 5-6: Climate and land cover niche (BERDL optional)

In [ ]:
r5 = dict(label="L5: Climate niche (WorldClim)", beta=np.nan, SE=np.nan, p=np.nan, n=np.nan, lam=np.nan)
r6 = dict(label="L6: Land cover niche (ESA CCI)", beta=np.nan, SE=np.nan, p=np.nan, n=np.nan, lam=np.nan)

if not _SPARK_AVAILABLE:
    print("Spark unavailable — skipping levels 5 and 6")
else:
    import pyspark.sql.functions as F
    try:
        tables_env = [r.tableName for r in _spark.sql("SHOW TABLES IN arkinlab.envdbs").collect()]
        has_wc  = "worldclim_master"            in tables_env
        has_esa = "global_landcover_esa_2_0deg"  in tables_env
        has_pop = "global_population_density"     in tables_env
        has_ee  = "earthenv_master"               in tables_env
        print(f"Tables: worldclim={has_wc}  esa={has_esa}  popdens={has_pop}  earthenv={has_ee}")
    except Exception as e:
        print(f"Table discovery failed: {e}")
        has_wc = has_esa = False

    # Level 5: WorldClim
    if has_wc:
        try:
            wc_schema = _spark.sql("DESCRIBE arkinlab.envdbs.worldclim_master").toPandas()
            print("WorldClim columns:", wc_schema["col_name"].tolist()[:20])
            # Quick sample to understand schema
            wc_samp = _spark.sql("SELECT * FROM arkinlab.envdbs.worldclim_master LIMIT 3").toPandas()
            wc_samp.attrs = {}
            print(wc_samp)
            # TODO: implement spatial join + niche computation after inspecting schema
            print("WorldClim table found — full implementation after schema inspection")
        except Exception as e:
            print(f"WorldClim error: {e}")
    else:
        print("worldclim_master not in arkinlab.envdbs — L5 skipped")

    # Level 6: ESA CCI land cover
    if has_esa:
        try:
            esa_schema = _spark.sql("DESCRIBE arkinlab.envdbs.global_landcover_esa_2_0deg").toPandas()
            print("ESA CCI columns:", esa_schema["col_name"].tolist()[:20])
            esa_samp = _spark.sql("SELECT * FROM arkinlab.envdbs.global_landcover_esa_2_0deg LIMIT 3").toPandas()
            esa_samp.attrs = {}
            print(esa_samp)
        except Exception as e:
            print(f"ESA CCI error: {e}")
    else:
        print("global_landcover_esa_2_0deg not in arkinlab.envdbs — L6 skipped")


## Compile and compare all PGLS results

In [ ]:
results = [r0a, r0b, r1, r2, r3, r4, r5, r6]
comp = pd.DataFrame(results)
comp["ci_lo"] = comp["beta"] - 1.96 * comp["SE"]
comp["ci_hi"] = comp["beta"] + 1.96 * comp["SE"]
comp["sig"] = comp["p"].apply(lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns")))
comp.to_csv(DATA / "37_niche_level_comparison.csv", index=False)

print("=== Niche breadth PGLS comparison ===")
for _, row in comp.iterrows():
    skip = "SKIP" if np.isnan(row["beta"]) else f"beta={row['beta']:+.4f}  p={row['p']:.3e}  n={row['n']}  {row['sig']}"
    print(f"  {row['label']}: {skip}")


## USA per-metal niche PGLS

In [ ]:
metal_results = []
for col in NGSA_ICP:
    # Strip method suffix to get metal name
    metal = col.split("_ICP")[0]
    sub = merge_p1(env_ngsa[["genus_lower", col]].dropna(subset=[col]))
    sub["niche"] = z_score(sub[col])
    sub = sub.dropna(subset=["niche"])
    if len(sub) < 60:
        continue
    res = run_pgls(sub, str(TREE_BAC), response="ko_per_mb_primary",
                   predictors=["niche"], taxon_col="genus_lower",
                   label=f"L4_{metal}", min_n=30)
    r = extract_pgls(res, metal)
    r["metal"] = metal; r["n_genera"] = len(sub)
    metal_results.append(r)
    print(f"  {metal:4s}: beta={r['beta']:+.4f}  p={r['p']:.3e}  n={r['n_genera']}  {'*' if r['p']<0.05 else 'ns'}")

metal_df = pd.DataFrame(metal_results).sort_values("beta")
metal_df["ci_lo"] = metal_df["beta"] - 1.96 * metal_df["SE"]
metal_df["ci_hi"] = metal_df["beta"] + 1.96 * metal_df["SE"]
metal_df["sig"] = metal_df["p"].apply(lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns")))
metal_df.to_csv(DATA / "37_usa_per_metal_niche.csv", index=False)


## Figure 1: Niche level comparison forest plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plot_df = comp.dropna(subset=["beta"]).copy().reset_index(drop=True)
n_rows  = len(plot_df)

fig, ax = plt.subplots(figsize=(FIGW["2col"], max(ROW_H * 0.8, n_rows * 0.58 + 0.9)))
grid_h(ax)

# First pass: draw all errorbars so xlim stabilises
for i, (_, row) in enumerate(plot_df.iterrows()):
    col = PALETTE[0] if row["beta"] < 0 else PALETTE[1]
    ax.errorbar(row["beta"], i,
                xerr=[[row["beta"] - row["ci_lo"]], [row["ci_hi"] - row["beta"]]],
                fmt="o", ms=6, color=col, ecolor=col, capsize=3, lw=1.2)

# Add reference lines
ax.axvline(0, color="gray", lw=0.8, ls="--")
ax.axvline(P1_BETA, color="black", lw=1.0, ls=":", alpha=0.7)

# Second pass: annotations at stable xlim
ax.relim(); ax.autoscale_view()
xl = ax.get_xlim()
x_ann = xl[1] + (xl[1] - xl[0]) * 0.01
for i, (_, row) in enumerate(plot_df.iterrows()):
    pstr = "<0.001" if row["p"] < 0.001 else f"={row['p']:.3f}"
    nstr = f"n={int(row['n'])}" if not np.isnan(row["n"]) else ""
    sig  = row["sig"] if row["sig"] != "ns" else ""
    ax.text(x_ann, i, f"{sig} p{pstr}  {nstr}", va="center", fontsize=7, color="#555555",
            clip_on=False)

ax.set_xlim(xl[0], xl[1] + (xl[1] - xl[0]) * 0.35)   # leave room for annotations
ax.set_yticks(range(n_rows))
ax.set_yticklabels(plot_df["label"].tolist(), fontsize=7.5)
ax.invert_yaxis()
ax.set_xlabel("PGLS β  (ko_per_mb_primary ~ niche breadth PC1)", fontsize=9)

bp = mpatches.Patch(color=PALETTE[0], label="β < 0 (expected)")
op = mpatches.Patch(color=PALETTE[1], label="β > 0 (unexpected)")
pl = plt.Line2D([0],[0], color="black", lw=1.0, ls=":", label=f"P1 ref β={P1_BETA}")
ax.legend(handles=[bp, op, pl], fontsize=7, loc="lower right")

fig.suptitle("NB37: Metal-gene density ~ niche breadth by environmental level", y=1.02, fontsize=11, fontweight="bold")
save(fig, FIGS / "fig_nb37_niche_level_comparison")
print("Saved fig_nb37_niche_level_comparison.pdf")


## Figure 2: USA per-metal niche breadth PGLS

In [ ]:
if len(metal_df) > 0:
    n_m = len(metal_df)
    fig2, ax2 = plt.subplots(figsize=(FIGW["2col"], max(ROW_H * 0.8, n_m * 0.52 + 0.9)))
    grid_h(ax2)

    for i, (_, row) in enumerate(metal_df.iterrows()):
        col2 = PALETTE[0] if row["beta"] < 0 else PALETTE[1]
        ax2.errorbar(row["beta"], i,
                     xerr=[[row["beta"] - row["ci_lo"]], [row["ci_hi"] - row["beta"]]],
                     fmt="o", ms=5.5, color=col2, ecolor=col2, capsize=3, lw=1.2)

    ax2.axvline(0, color="gray", lw=0.8, ls="--")

    # Annotations after xlim stabilises
    ax2.relim(); ax2.autoscale_view()
    xl2 = ax2.get_xlim()
    x_ann2 = xl2[1] + (xl2[1] - xl2[0]) * 0.01
    for i, (_, row) in enumerate(metal_df.iterrows()):
        sig2 = row["sig"] if row["sig"] != "ns" else ""
        pstr2 = "<0.001" if row["p"] < 0.001 else f"={row['p']:.3f}"
        ax2.text(x_ann2, i, f"{sig2} p{pstr2}", va="center", fontsize=7.5, clip_on=False)

    ax2.set_xlim(xl2[0], xl2[1] + (xl2[1] - xl2[0]) * 0.35)
    ax2.set_yticks(range(n_m))
    ax2.set_yticklabels([str(r["metal"]) for _, r in metal_df.iterrows()], fontsize=8.5)
    ax2.invert_yaxis()
    ax2.set_xlabel("PGLS β  (ko_per_mb_primary ~ metal niche breadth SD)", fontsize=9)
    ax2.set_ylabel("Metal (USGS NGSA ICP-MS)", fontsize=9)
    ax2.set_title("USA geochemical niche breadth by individual metal", fontsize=10)
    fig2.suptitle("NB37: Per-Metal Niche Breadth — USA (USGS NGSA)", y=1.02, fontsize=11, fontweight="bold")
    save(fig2, FIGS / "fig_nb37_usa_metal_niche")
    print("Saved fig_nb37_usa_metal_niche.pdf")
else:
    print("No per-metal results to plot")


## Summary

In [ ]:
print("=== NB37 SUMMARY ===")
print(f"Niche levels tested: {len(comp)}")
sig_df = comp[comp["p"] < 0.05]
print(f"Significant associations (p<0.05): {len(sig_df)}")
if len(sig_df):
    print(sig_df[["label","beta","p","n"]].to_string(index=False))
neg_sig = comp[(comp["p"] < 0.05) & (comp["beta"] < 0)]
print(f"Significant in expected direction (beta < 0): {len(neg_sig)}")
if len(neg_sig):
    print(neg_sig[["label","beta","p"]].to_string(index=False))

if len(metal_df) > 0:
    print(f"USA per-metal: {len(metal_df)} metals tested")
    sig_metals = metal_df[metal_df["p"] < 0.05]
    print(f"Significant metals: {list(sig_metals['metal'])}")
